# CICIDS2017 Preprocessing

## Objective

Prepare the CICIDS2017 dataset for multiclass intrusion detection while preserving the original raw dataset.

### Preprocessing goals

- Normalize column names
- Validate the dataset schema
- Handle missing values
- Handle infinite values
- Define a duplicate-row policy
- Prepare multiclass target labels
- Prevent data leakage
- Prepare data for model training

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
DATA_DIR =  Path("../data/raw/MachineLearningCVE")

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

for file in csv_files:
    print(file.name)

Found 8 CSV files
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


**Schema validation.**

In [3]:
# Check the number of columns  in each file

for file in csv_files:
    temp = pd.read_csv(file)
    temp.columns = temp.columns.str.strip()

    print(f"{file.name}: {len(temp.columns)} columns")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 79 columns
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 79 columns
Friday-WorkingHours-Morning.pcap_ISCX.csv: 79 columns
Monday-WorkingHours.pcap_ISCX.csv: 79 columns
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79 columns
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 79 columns
Tuesday-WorkingHours.pcap_ISCX.csv: 79 columns
Wednesday-workingHours.pcap_ISCX.csv: 79 columns


In [4]:
# compair schemas

reference = None

for file in csv_files:
    temp = pd.read_csv(file)
    temp.columns = temp.columns.str.strip()

    if reference is None:
        reference = list(temp.columns)
        print(f"Reference schema: {file.name}")
    else:
        if list(temp.columns)== reference:
            print(f"{file.name}: schema matches")
        else:
            print(f"{file.name}: schema DOES NOT matches" )



Reference schema: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: schema matches
Friday-WorkingHours-Morning.pcap_ISCX.csv: schema matches
Monday-WorkingHours.pcap_ISCX.csv: schema matches
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: schema matches
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: schema matches
Tuesday-WorkingHours.pcap_ISCX.csv: schema matches
Wednesday-workingHours.pcap_ISCX.csv: schema matches


In [5]:
all_dfs = []

for file in csv_files:
    temp = pd.read_csv(file)
    temp.columns = temp.columns.str.strip()

    #keep the source from auditing
    temp["Source_File"] = file.name

    all_dfs.append(temp)

df_all = pd.concat(all_dfs, ignore_index = True)
print("Combined shape", df_all.shape)

Combined shape (2830743, 80)


In [6]:
print(df_all["Label"].value_counts())

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [7]:
class_distribution = df_all["Label"].value_counts(normalize=True) * 100

print(class_distribution)

Label
BENIGN                        80.300366
DoS Hulk                       8.162981
PortScan                       5.614427
DDoS                           4.522735
DoS GoldenEye                  0.363615
FTP-Patator                    0.280421
SSH-Patator                    0.208320
DoS slowloris                  0.204752
DoS Slowhttptest               0.194260
Bot                            0.069452
Web Attack � Brute Force       0.053237
Web Attack � XSS               0.023033
Infiltration                   0.001272
Web Attack � Sql Injection     0.000742
Heartbleed                     0.000389
Name: proportion, dtype: float64


In [8]:
print("Number of classes:", df_all["Label"].nunique())

Number of classes: 15


In [9]:
print(df_all["Label"].unique())

<StringArray>
[                    'BENIGN',                       'DDoS',
                   'PortScan',                        'Bot',
               'Infiltration',   'Web Attack � Brute Force',
           'Web Attack � XSS', 'Web Attack � Sql Injection',
                'FTP-Patator',                'SSH-Patator',
              'DoS slowloris',           'DoS Slowhttptest',
                   'DoS Hulk',              'DoS GoldenEye',
                 'Heartbleed']
Length: 15, dtype: str


In [10]:
# clean lable
df_all["Label_Clean"] = df_all["Label"].replace({
    "Web Attack � Brute Force": "Web Attack Brute Force",
    "Web Attack � XSS": "Web Attack XSS",
    "Web Attack � Sql Injection": "Web Attack SQL Injection"
})

In [11]:
print(df_all[["Label", "Label_Clean"]].drop_duplicates())

                              Label               Label_Clean
0                            BENIGN                    BENIGN
18883                          DDoS                      DDoS
227208                     PortScan                  PortScan
536284                          Bot                       Bot
1299546                Infiltration              Infiltration
1534402    Web Attack � Brute Force    Web Attack Brute Force
1593899            Web Attack � XSS            Web Attack XSS
1612058  Web Attack � Sql Injection  Web Attack SQL Injection
1703478                 FTP-Patator               FTP-Patator
1854120                 SSH-Patator               SSH-Patator
2144598               DoS slowloris             DoS slowloris
2207315            DoS Slowhttptest          DoS Slowhttptest
2212900                    DoS Hulk                  DoS Hulk
2469073               DoS GoldenEye             DoS GoldenEye
2735170                  Heartbleed                Heartbleed


In [12]:
print("Classes:", df_all["Label_Clean"].nunique())

Classes: 15


In [13]:
label_counts = df_all["Label_Clean"].value_counts()

print(label_counts)
print("\nNumber of classes:", df_all["Label_Clean"].nunique())

Label_Clean
BENIGN                      2273097
DoS Hulk                     231073
PortScan                     158930
DDoS                         128027
DoS GoldenEye                 10293
FTP-Patator                    7938
SSH-Patator                    5897
DoS slowloris                  5796
DoS Slowhttptest               5499
Bot                            1966
Web Attack Brute Force         1507
Web Attack XSS                  652
Infiltration                     36
Web Attack SQL Injection         21
Heartbleed                       11
Name: count, dtype: int64

Number of classes: 15


In [14]:
print("Missing labels:", df_all["Label_Clean"].isna().sum())

Missing labels: 0


In [15]:
print("\nUnique labels:")
for label in sorted(df_all["Label_Clean"].unique()):
    print("-", label)


Unique labels:
- BENIGN
- Bot
- DDoS
- DoS GoldenEye
- DoS Hulk
- DoS Slowhttptest
- DoS slowloris
- FTP-Patator
- Heartbleed
- Infiltration
- PortScan
- SSH-Patator
- Web Attack Brute Force
- Web Attack SQL Injection
- Web Attack XSS


In [16]:
# missing values
missing_counts = df_all.isna().sum()\


print("Columns with missing values")
print(missing_counts[missing_counts > 0])

Columns with missing values
Flow Bytes/s    1358
dtype: int64


In [17]:
# infinite values
numaric_cols = df_all.select_dtypes(include=np.number).columns

infinite_counts = np.isinf(df_all[numaric_cols]).sum()

print("\n Columns with infinite  values")
print(infinite_counts[infinite_counts > 0])


 Columns with infinite  values
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64


In [18]:
# Count duplicate rows
duplicate_count = df_all.duplicated().sum()

print("Total duplicate rows:", duplicate_count)

Total duplicate rows: 256479


In [19]:
# Exclude Source_File from duplicate checking
feature_cols = [
    col for col in df_all.columns
    if col != "Source_File"
]

# Count duplicate flow records
duplicate_count = df_all.duplicated(
    subset=feature_cols
).sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 308381


In [ ]:
# Check whether duplicate feature rows have different labels
duplicate_groups = df_all[
    df_all.duplicated(subset=feature_cols, keep=False)
]

label_variation = duplicate_groups.groupby(feature_cols)["Label_Clean"].nunique()

print("Duplicate groups with multiple labels:",
      (label_variation > 1).sum())

In [ ]:
# Count duplicate rows by attack class
duplicate_label_counts = (
    duplicate_groups["Label_Clean"]
    .value_counts()
)

print(duplicate_label_counts)

Label_Clean
BENIGN                    236257
PortScan                  101501
DoS Hulk                   59564
SSH-Patator                 2826
FTP-Patator                 2457
DoS slowloris                507
DoS Slowhttptest             323
Web Attack Brute Force        62
DDoS                          20
Bot                           19
DoS GoldenEye                 14
Name: count, dtype: int64


In [ ]:
# Count each class in every source file
class_by_source = pd.crosstab(
    df_all["Source_File"],
    df_all["Label_Clean"]
)

print(class_by_source)

Label_Clean                                         BENIGN   Bot    DDoS  \
Source_File                                                                
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv     97718     0  128027   
Friday-WorkingHours-Afternoon-PortScan.pcap_ISC...  127537     0       0   
Friday-WorkingHours-Morning.pcap_ISCX.csv           189067  1966       0   
Monday-WorkingHours.pcap_ISCX.csv                   529918     0       0   
Thursday-WorkingHours-Afternoon-Infilteration.p...  288566     0       0   
Thursday-WorkingHours-Morning-WebAttacks.pcap_I...  168186     0       0   
Tuesday-WorkingHours.pcap_ISCX.csv                  432074     0       0   
Wednesday-workingHours.pcap_ISCX.csv                440031     0       0   

Label_Clean                                         DoS GoldenEye  DoS Hulk  \
Source_File                                                                   
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                0         0   
Fr

In [ ]:
# Count unique flow patterns
unique_flow_count = df_all[feature_cols].drop_duplicates().shape[0]

print("Total rows:", len(df_all))
print("Unique flow patterns:", unique_flow_count)
print("Duplicate rows:", len(df_all) - unique_flow_count)

Total rows: 2830743
Unique flow patterns: 2522362
Duplicate rows: 308381


In [ ]:
# Create an ID for each unique flow pattern
df_all["Flow_Group"] = (
    df_all[feature_cols]
    .astype(str)
    .agg("|".join, axis=1)
    .factorize()[0]
)

print("Unique flow groups:", df_all["Flow_Group"].nunique())